In [ ]:
import sys; sys.path.append("..")
import os
import json
import random
from functools import partial

from dotenv import load_dotenv
from huggingface_hub import login
from transformers import AutoTokenizer

from src.config import load_config

cfg = load_config("../config.yaml")

In [ ]:
load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")
if (HF_TOKEN == None):
    raise ValueError("HF_TOKEN is not set")
login(token=HF_TOKEN)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(cfg.model.base_model_id)

from src.tokens import get_special_tokens
tokens = get_special_tokens(tokenizer)

### Whatsapp txt into filtered CSV

In [ ]:
# chat cleaning logic (noise removal, latin-1 decoding) lives in src/data/whatsapp_parser.py
from src.data.whatsapp_parser import process_data

In [ ]:
chats = ["ludovica", "mamma", "genni", "benny", "cammisa"]
for chat in chats:
    process_data(f"../chat/{chat}.txt", f"../data/{chat}.csv", assistant_name=cfg.data.assistant_name)

File saved successfully in: data/ludovica.csv
File saved successfully in: data/mamma.csv
File saved successfully in: data/genni.csv
File saved successfully in: data/benny.csv
File saved successfully in: data/cammisa.csv


### CSV to json

In [ ]:
# role assignment and chat merging live in src/data/dataset_builder.py
from src.data.dataset_builder import process_conversation, create_dataset

In [ ]:
csv_files = [
    "../data/ludovica.csv",
    "../data/genni.csv",
    "../data/mamma.csv",
    "../data/cammisa.csv"
]

create_dataset(csv_files, "../dataset.json", assistant_name=cfg.data.assistant_name)

### json to dataset

In [ ]:
# special tokens live in src/tokens.py, prompt building in src/data/dataset_builder.py:
# context selected with a time-gap heuristic, consecutive same-role messages grouped in one turn
from src.data.dataset_builder import create_formatted_prompt, parse_timestamp
from src.prompts import TRAINING_SYSTEM_PROMPT

In [ ]:
from src.data.dataset_builder import create_dataset_list

In [ ]:
dataset_list = create_dataset_list(
    "../dataset.json",
    tokens=tokens,
    max_context_messages=cfg.data.max_context_messages,
    time_gap_seconds=cfg.data.time_gap_seconds,
)

# print some examples
if dataset_list:
    random_indices = random.sample(range(len(dataset_list)), 3)

    print(f"Generated {len(dataset_list)} training examples.\n")
    print("--- Randomly Selected Examples ---")
    for i, idx in enumerate(random_indices):
        example = dataset_list[idx]
        print(f"--- Example {i+1} (Index: {idx}) ---")
        print("Prompt:")
        print(example['prompt'])
        print("\nResponse:")
        print(example['response'])
        print("---------------------\n")
else:
    print("ERROR: No training examples were generated. The dataset_list is empty.")

Generated 14390 training examples.

--- Randomly Selected Examples ---
--- Example 1 (Index: 7984) ---
Prompt:
<｜begin▁of▁sentence｜>You are Francesco Brigante, a 22 years old Italian Computer Science student in Rome. 
Respond naturally as him in Italian, maintaining his characteristic communication style. 
Keep responses concise and contextual.

Continue this conversation with the User, who is a friend of Francesco:
<｜User｜>*manda un audio*
*manda un audio*<|turn_end|>
<｜Assistant｜>Difficile cumba
Sto ridendo troppo
Mado<|turn_end|>
<｜User｜>Perché<|turn_end|>
<｜Assistant｜>

Response:
Per quello che dici<｜end▁of▁sentence｜>
---------------------

--- Example 2 (Index: 2343) ---
Prompt:
<｜begin▁of▁sentence｜>You are Francesco Brigante, a 22 years old Italian Computer Science student in Rome. 
Respond naturally as him in Italian, maintaining his characteristic communication style. 
Keep responses concise and contextual.

Continue this conversation with the User, who is a friend of Francesco

In [ ]:
# 80% train, 10% val, 10% test
from src.data.dataset_builder import build_splits

train_dataset, val_dataset, test_dataset = build_splits(
    dataset_list, test_size=cfg.data.test_size, seed=cfg.data.seed
)

### Tokenization

In [ ]:
# tokenization with prompt masking (-100 labels on everything before the response)
from src.data.tokenization import tokenize_function

tokenize = partial(tokenize_function, tokenizer=tokenizer, max_length=cfg.data.max_length)

In [ ]:
# apply tokenization
tokenized_train = train_dataset.map(
    tokenize,
    batched=True,
    remove_columns=train_dataset.column_names
)

tokenized_val = val_dataset.map(
    tokenize,
    batched=True,
    remove_columns=val_dataset.column_names
)

tokenized_test = test_dataset.map(
    tokenize,
    batched=True,
    remove_columns=test_dataset.column_names
)

# save tokenized datasets
tokenized_train.save_to_disk('../datasets/tokenized_train')
tokenized_val.save_to_disk('../datasets/tokenized_val')
tokenized_test.save_to_disk('../datasets/tokenized_test')

Map:   0%|          | 0/11512 [00:00<?, ? examples/s]

[❌] Assistant token not found in input_ids.
[❌] Assistant token not found in input_ids.
[❌] Assistant token not found in input_ids.
[❌] Assistant token not found in input_ids.
[❌] Assistant token not found in input_ids.
[❌] Assistant token not found in input_ids.
[❌] Assistant token not found in input_ids.
[❌] Assistant token not found in input_ids.
[❌] Assistant token not found in input_ids.
[❌] Assistant token not found in input_ids.


Map:   0%|          | 0/1439 [00:00<?, ? examples/s]

Map:   0%|          | 0/1439 [00:00<?, ? examples/s]

[❌] Assistant token not found in input_ids.
[❌] Assistant token not found in input_ids.


Saving the dataset (0/1 shards):   0%|          | 0/11512 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1439 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1439 [00:00<?, ? examples/s]

### Check for correctness

In [ ]:
# check to see if the tokenized datasets are loaded correctly
from datasets import load_from_disk

tokenized_train = load_from_disk('../datasets/tokenized_train')
tokenized_val = load_from_disk('../datasets/tokenized_val')
tokenized_test = load_from_disk('../datasets/tokenized_test')

print(f"Training examples: {len(tokenized_train)}")
print(f"Validation examples: {len(tokenized_val)}")
print(f"Test examples: {len(tokenized_test)}")

Training examples: 11512
Validation examples: 1439
Test examples: 1439


In [ ]:
from src.data.tokenization import print_tokenized_example

print_tokenized_example(tokenized_train, tokenizer)

--- Example index: 5582 ---
== INPUT IDS ==
[151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151

In [ ]:
# checks that attention mask matches padding and labels mask exactly the prompt
from src.data.tokenization import verify_tokenized_example

for example in tokenized_train:
    if not verify_tokenized_example(example, tokenizer):
        print("Error in tokenized example!")
        break

[✅] Attention mask matches padding: 148.
[✅] Labels correctly mask prompt tokens, 12 response tokens detected.
[✅] Attention mask matches padding: 169.
[✅] Labels correctly mask prompt tokens, 6 response tokens detected.
[✅] Attention mask matches padding: 176.
[✅] Labels correctly mask prompt tokens, 7 response tokens detected.
[✅] Attention mask matches padding: 139.
[✅] Labels correctly mask prompt tokens, 3 response tokens detected.
[✅] Attention mask matches padding: 172.
[✅] Labels correctly mask prompt tokens, 9 response tokens detected.
[✅] Attention mask matches padding: 141.
[✅] Labels correctly mask prompt tokens, 5 response tokens detected.
[✅] Attention mask matches padding: 145.
[✅] Labels correctly mask prompt tokens, 11 response tokens detected.
[✅] Attention mask matches padding: 123.
[✅] Labels correctly mask prompt tokens, 25 response tokens detected.
[✅] Attention mask matches padding: 140.
[✅] Labels correctly mask prompt tokens, 7 response tokens detected.
[✅] Att

In [ ]:
# percentage of examples with no padding, useful to set the batch size
from src.data.tokenization import count_full_attention

count_full_attention(tokenized_train)

✅ Percentage of examples with full attention (no padding): 92 / 11512 = 0.80%


0.7991660875608061